In [1]:
import os 
data_raw_path = 'data/data_raw'
dirs = [x for x in os.listdir(data_raw_path) if not x.startswith('.')]

for dirc in dirs:
    dir_raw_path = data_raw_path + '/' + dirc
    #os.filepaths


In [2]:
data_raw_path = 'data/data_raw'

file_paths = [dirpath+'/'+filename for dirpath, dirnames, filenames in os.walk(data_raw_path) for filename in filenames ]
#for dirpath, dirnames, filenames in os.walk(data_raw_path):
    #print(f"Current Folder: {dirpath}")
   ## print(f"Subdirectories: {dirnames}")
   # print(f"Files: {filenames}")
   # print("-" * 20)
print(file_paths)

['data/data_raw/2018/session_11_Q/laps.csv', 'data/data_raw/2018/session_11_Q/drivers.csv', 'data/data_raw/2018/session_11_Q/total_laps.csv', 'data/data_raw/2018/session_11_Q/track_status.csv', 'data/data_raw/2018/session_11_Q/race_control_messages.csv', 'data/data_raw/2018/session_11_Q/event.csv', 'data/data_raw/2018/session_11_Q/weather_data.csv', 'data/data_raw/2018/session_11_Q/session_status.csv', 'data/data_raw/2018/session_11_Q/results.csv', 'data/data_raw/2018/session_11_Q/session_info.csv', 'data/data_raw/2018/session_4_Q/laps.csv', 'data/data_raw/2018/session_4_Q/drivers.csv', 'data/data_raw/2018/session_4_Q/total_laps.csv', 'data/data_raw/2018/session_4_Q/track_status.csv', 'data/data_raw/2018/session_4_Q/race_control_messages.csv', 'data/data_raw/2018/session_4_Q/event.csv', 'data/data_raw/2018/session_4_Q/weather_data.csv', 'data/data_raw/2018/session_4_Q/session_status.csv', 'data/data_raw/2018/session_4_Q/results.csv', 'data/data_raw/2018/session_4_Q/session_info.csv', '

In [3]:
from pyspark.sql.functions import input_file_name, regexp_extract, col
from pyspark.sql import SparkSession

spark = (SparkSession
         .builder
         .appName('SparkDBApp')
         .getOrCreate())

df = spark.read.csv(
    "data/data_raw/*/session_*/drivers.csv",
    header=True,
    inferSchema=True
)
drivers = df.withColumn(
    "session_number",
    regexp_extract(input_file_name(), r"session_(\d+)", 1).cast("int")
)

#drivers.show(50)
file_names = ['drivers', 
              #'event',
              'laps',
              'race_control_messages',
              'results',
              'session_info',
              'session_status',
              'track_status',
              'weather_data']
def load_files(file_name):
    df = spark.read.csv(
        f"data/data_raw/*/session_*/{file_name}.csv",
        header=True,
        inferSchema=True
    )

    file_path = input_file_name()
    df = (
        df
        .withColumn('source_file', file_path)
        .withColumn(
            'year',
            regexp_extract(col('source_file'), r'data_raw/(\d{4})/', 1).cast('int')
        )
        .withColumn(
            'session_number',
            regexp_extract(col('source_file'), r'session_(\d+)', 1).cast('int')
        )
        .withColumn('type_of_race',
                   regexp_extract(col('source_file'), r'session_\d+_(\S)', 1).cast('string')
                   )
        .drop('_c0')
    )

    return df

load_files('drivers').show(50)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/11 11:31:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/11 11:31:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/11 11:31:56 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/data_raw/*/session_*/drivers.csv.
java.io.FileNotFoundException: File data/data_raw/*/session_*/drivers.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.ap

+---+--------------------+----+--------------+------------+
|  0|         source_file|year|session_number|type_of_race|
+---+--------------------+----+--------------+------------+
| 63|file:///sfs/gpfs/...|2026|             1|           Q|
| 12|file:///sfs/gpfs/...|2026|             1|           Q|
|  6|file:///sfs/gpfs/...|2026|             1|           Q|
| 16|file:///sfs/gpfs/...|2026|             1|           Q|
| 81|file:///sfs/gpfs/...|2026|             1|           Q|
|  1|file:///sfs/gpfs/...|2026|             1|           Q|
| 44|file:///sfs/gpfs/...|2026|             1|           Q|
| 30|file:///sfs/gpfs/...|2026|             1|           Q|
| 41|file:///sfs/gpfs/...|2026|             1|           Q|
|  5|file:///sfs/gpfs/...|2026|             1|           Q|
| 27|file:///sfs/gpfs/...|2026|             1|           Q|
| 87|file:///sfs/gpfs/...|2026|             1|           Q|
| 31|file:///sfs/gpfs/...|2026|             1|           Q|
| 10|file:///sfs/gpfs/...|2026|         

In [4]:
#spark.sql("CREATE DATABASE learn_spark_db")
#spark.sql("USE learn_spark_db")
#spark.sql("CREATE TABLE managed_us_delay_flights_tbl (date STRING, delay INT,  distance INT, origin STRING, destination STRING)")
spark.sql('DROP database IF EXISTS formula1')
spark.sql('CREATE database IF NOT EXISTS formula1')
spark.sql('USE formula1')

DataFrame[]

In [5]:
drivers = load_files("drivers")
laps = load_files("laps")
results = load_files("results")
session_info = load_files("session_info")
session_status = load_files("session_status")
track_status = load_files("track_status")
weather_data = load_files("weather_data")

26/07/11 11:32:02 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/data_raw/*/session_*/drivers.csv.
java.io.FileNotFoundException: File data/data_raw/*/session_*/drivers.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache

In [6]:
drivers.write.mode("overwrite").saveAsTable("formula1.drivers")
laps.write.mode("overwrite").saveAsTable("formula1.laps")
results.write.mode("overwrite").saveAsTable("formula1.results")
session_info.write.mode("overwrite").saveAsTable("formula1.session_info")
session_status.write.mode("overwrite").saveAsTable("formula1.session_status")
track_status.write.mode("overwrite").saveAsTable("formula1.track_status")
weather_data.write.mode("overwrite").saveAsTable("formula1.weather_data")

SparkRuntimeException: [LOCATION_ALREADY_EXISTS] Cannot name the managed table as `spark_catalog`.`formula1`.`drivers`, as its associated location 'file:/sfs/gpfs/tardis/home/azd9vv/Formula-1/spark-warehouse/formula1.db/drivers' already exists. Please pick a different table name, or remove the existing location first. SQLSTATE: 42710

In [ ]:
import pyspark.sql.functions as F
#event_long.select('field').distinct().show()

In [ ]:
spark.sql('SHOW TABLES').show()

In [ ]:
spark.sql('USE formula1')

In [ ]:
spark.sql('Select * From formula1.drivers').show()

In [ ]:
drivers = spark.table('formula1.drivers')
drivers_fixed = drivers.withColumnRenamed("0", "driver_number")

drivers_fixed.write.mode("overwrite").saveAsTable("drivers_fixed")

spark.sql("DROP TABLE drivers")
spark.sql("ALTER TABLE drivers_fixed RENAME TO drivers")

In [ ]:
spark.sql('Select * From formula1.drivers').show()

In [ ]:
laps = spark.table('laps')
laps.show()

In [ ]:
laps.select('LapTime').show(5)

In [ ]:
0 

In [ ]:
from pyspark.sql.functions import col, regexp_extract, when, length

laps_clean_laptime = laps\
    .withColumn('laptime_days', 
                when(
                    length(regexp_extract(col("LapTime"), r"(\d+) days", 1)) > 0,
                    regexp_extract(col('LapTime'), r'(\d+) days', 1).cast('int')
                )
               )\
    .withColumn('laptime_hours',
                when(
                    length(regexp_extract(col("LapTime"), r'days (\d{2}):', 1)) > 0,
                    regexp_extract(col('LapTime'), r'days (\d{2}):', 1).cast('int')
                )
                )\
    .withColumn('laptime_minutes',
                when(
                    length(regexp_extract(col('LapTime'), r'days \d{2}:(\d{2}):', 1)) > 0,
                    regexp_extract(col('LapTime'), r'days \d{2}:(\d{2}):', 1).cast('int')
                )
                )\
    .withColumn('laptime_seconds',
                when(
                    length(regexp_extract(col('LapTime'), r'days \d{2}:\d{2}:(\d{2}.\d+)', 1)) > 0,
                    regexp_extract(col('LapTime'), r'days \d{2}:\d{2}:(\d{2}.\d+)', 1).cast('double')
                )
                )\
    .withColumn('total_laptime_secs',
                col('laptime_days')*60*3+col('laptime_hours')*60*2+col('laptime_minutes')*60+col('laptime_seconds'))\
    .drop('laptime_hours', 'laptime_minutes', 'laptime_seconds')
laps_clean_laptime.select('total_laptime_secs').show(5)

In [ ]:
spark.sql('USE formula1')

In [ ]:
laps_clean_laptime.write.mode("overwrite").saveAsTable("laps_time_fixed2")

spark.sql('DROP TABLE laps')
spark.sql('ALTER TABLE laps_time_fixed2 RENAME TO laps')

In [ ]:
laps = spark.sql('SELECT * from formula1.laps')

In [ ]:
laps.printSchema()

In [ ]:
import pyspark.sql.functions as F
some_laps = laps.filter((col('year')==2021) &(col('session_number')==1) & (col('total_laptime_secs').isNotNull())).groupBy('Team', 'Driver', 'DriverNumber').agg(F.mean('total_laptime_secs').alias('Average Lap Time'))
some_laps.show()


In [ ]:
laps.printSchema()
res = spark.sql('Select * from formula1.results')
res.show(10)

In [ ]:
res.printSchema()

In [ ]:
some_res = res.filter((col('session_number')==1) & (col('year')==2021) & (col('Points').isNotNull())& (col('type_of_race')=='R')).select('TeamName', 'DriverNumber', 'Points').orderBy('Points')#.show(30)

In [ ]:
join_res = some_res.join(some_laps, how='left', on='DriverNumber')
join_res.show(20)

In [ ]:
join_res.plot(y='Points', x='Average Lap Time')

In [ ]:
# lets get the average lap position

some_pos = laps.filter((col('session_number')==1) & (col('year')==2021)).groupby('DriverNumber', 'Team','type_of_race').agg(F.mean('Position').alias('Average Position'))#.select('DriverNumber', 'Team', 'Position
some_pos.show(20)

In [ ]:
laps.groupby('Driver', 'type_of_race').agg(F.mean('Position').alias('Average Position'), F.mean('total_laptime_secs').alias('Average Laptime'),F.countDistinct("year").alias("Years Raced")).plot.scatter(x='Average Position', y='Average Laptime',color='Driver',size='Years Raced')#.show()

In [ ]:
# do drivers who win qualifying win the actual race?

#who won qualifying?
won_qualifying = res.filter((col('type_of_race')=='Q') & (col('Position')==1)).groupBy('DriverId').agg(F.countDistinct('Position').alias('Position Count')).orderBy('Position Count', ascending=False)
won_qualifying.show(50)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# clean position type in case it was read as string
res_clean = res.withColumn("Position_int", col("Position").cast("int"))

# who won qualifying / got pole?
qual_winners = (
    res_clean
    .filter((col("type_of_race") == "Q") & (col("Position_int") == 1))
    .select(
        "year",
        "session_number",
        col("DriverId").alias("qualifying_winner")
    )
    .dropDuplicates(["year", "session_number"])
)

# who won the race?
race_winners = (
    res_clean
    .filter((col("type_of_race") == "R") & (col("Position_int") == 1))
    .select(
        "year",
        "session_number",
        col("DriverId").alias("race_winner")
    )
    .dropDuplicates(["year", "session_number"])
)

# compare qualifying winner vs race winner
pole_vs_race = (
    qual_winners
    .join(race_winners, on=["year", "session_number"], how="inner")
    .withColumn(
        "qualifying_winner_won_race",
        col("qualifying_winner") == col("race_winner")
    )
)

pole_vs_race.show(50)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

plot_df = (
    pole_vs_race
    .groupBy("qualifying_winner_won_race")
    .agg(F.count("*").alias("count"))
    .withColumn(
        "result",
        F.when(col("qualifying_winner_won_race") == True, "Pole winner won race")
         .otherwise("Pole winner did not win race")
    )
    .select("result", "count")
    .toPandas()
)

ax = plot_df.plot.bar(
    x="result",
    y="count",
    legend=False,
    rot=0,
    figsize=(8, 5)
)

ax.set_xlabel("")
ax.set_ylabel("Number of races")
ax.set_title("Did the qualifying winner win the race?")